<a href="https://colab.research.google.com/github/Aditya-Bang/MinesweeperGPT/blob/main/tests/finetuning/MinesweeperGPT_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi
!nvcc --version
!uname -a

Mon Sep 15 23:29:52 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%capture
!pip install -qqq torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

In [3]:
%%capture
!pip install -qqq unsloth==2025.6.2 unsloth_zoo==2025.6.1 trl==0.18.1 vllm==0.8.5.post1 xformers==0.0.29.post2 triton==3.2.0 accelerate==1.7.0 transformers==4.51.3 torchao==0.12.0 wandb

In [4]:
!pip list | grep -E 'torch|triton|vllm|unsloth|transformers|xformers|accelerate|trl|wandb'

accelerate                               1.7.0
fastrlock                                0.8.3
sentence-transformers                    5.1.0
torch                                    2.6.0+cu124
torchao                                  0.12.0
torchaudio                               2.6.0+cu124
torchdata                                0.11.0
torchsummary                             1.5.1
torchtune                                0.6.1
torchvision                              0.21.0+cu124
transformers                             4.51.3
triton                                   3.2.0
trl                                      0.18.1
unsloth                                  2025.6.2
unsloth_zoo                              2025.6.1
vllm                                     0.8.5.post1
wandb                                    0.21.3
xformers                                 0.0.29.post2


In [5]:
%%capture
!unzip train-data.zip -d train-data
!unzip test-data.zip -d test-data

In [6]:
import torch
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 09-15 23:35:20 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 09-15 23:35:21 [__init__.py:239] Automatically detected platform cuda.


In [7]:
max_seq_length = 512  # Can increase for longer reasoning traces
lora_rank = 32         # Larger rank = smarter, but slower

# Load model + tokenizer with vLLM acceleration
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B",
    max_seq_length = max_seq_length,
    load_in_4bit = True,       # False for LoRA 16bit
    fast_inference = True,      # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.45, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,  # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],  # Remove QKVO if out of memory
    lora_alpha=lora_rank*2,
    use_gradient_checkpointing="unsloth",  # Enable long context finetuning
    random_state=3407,
)

==((====))==  Unsloth 2025.6.2: Fast Qwen3 patching. Transformers: 4.51.3. vLLM: 0.8.5.post1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen3-4b-unsloth-bnb-4bit with actual GPU utilization = 44.57%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.74 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 512. Num Sequences = 160.
Unsloth: vLLM's KV Cache can use up to 3.82 GB. Also swap space = 0 GB.
WARNING 09-15 23:35:33 [config.py:2972] Casting torch.bfloat16 to torch.float16.
INFO 09-15 23:36:03 [config.py:717] This model supports multiple tasks: {'reward', 'classify', 'embed', 'generate', '

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth 2025.6.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [8]:
# src/models.py
from dataclasses import dataclass
from typing import List

@dataclass
class MinesweeperExample:
    input: str
    board_state: List[List[str]]
    hidden_state: List[List[str]]

@dataclass
class Move:
    row: int
    col: int
    action: str

# src/utils.py
from pathlib import Path

def get_base_directory() -> Path:
    """
    Returns the base directory path of the project.
    """
    return Path.cwd()

# src/globals.py
TRAINING_ROWS = 5
TRAINING_COLS = 5
TRAINING_MIN_MINES = 5
TRAINING_MAX_MINES = 8

# src/finetuning/prompt.py
SYSTEM_PROMPT = f"""
You are a Minesweeper assistant.
The game board is always {TRAINING_ROWS}x{TRAINING_COLS} in size.
You will be given ONLY the current board state as input from the user.

Your task: Suggest exactly ONE valid next move for the minesweeper board given by the user.

Move format rules (must follow exactly one of these two):
1. "row: NUM, col: NUM, action: reveal"       → to reveal a cell
2. "row: NUM, col: NUM, action: flag"         → to flag a cell as a mine

Board representation:
- '*' means the tile has not been revealed yet.
- Numbers 0–8 show how many mines are adjacent to that square.
- 'F' means the tile has already been flagged as a mine.
- The board will be displayed as a grid of symbols only.

Important condition:
- You may only suggest to reveal of flag a tile that is not already revealed, i.e. contains '*'.
- Do NOT suggest moves on numbers or flagged tiles, as these have already been revealed or correctly flagged.

Summary:
- Suggest one valid move next with the format "row: NUM, col: NUM, action: reveal" or "row: NUM, col: NUM, action: flag", where NUM is an integer in the range [1, {TRAINING_ROWS}] for rows and [1, {TRAINING_COLS}] for columns.
- Only output the valid move.
"""

def add_row_numbers(board_str: str) -> str:
    lines = board_str.splitlines()
    numbered_lines = [f"Row {i}: {line}" for i, line in enumerate(lines, start=1) if line.strip()]
    return "\n".join(numbered_lines)

def format_example(board: str) -> dict:
    formatted_board: str = add_row_numbers(board)
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": formatted_board + "\n/no_think"},
    ]

# src/finetuning/dataset.py
from pathlib import Path
from typing import List, Dict, Any
import random
from datasets import Dataset


class MinesweeperDatasetLoader:
    def __init__(self, data_dir: str = "data"):
        self.base_dir: Path = get_base_directory()
        self.data_dir: Path = (self.base_dir / data_dir).resolve()
        print(f"Using data directory: {self.data_dir}")
        if not self.data_dir.exists():
            print(f"Data directory does not exist: {self.data_dir}")
        else:
            print(f"Data directory exists: {self.data_dir}")
        self.games = sorted([g for g in self.data_dir.glob("game*") if g.is_dir()])
        if not self.games:
            print(f"No game directories found in {self.data_dir}")
        else:
            print(f"Found {len(self.games)} game directories in {self.data_dir}")

    def load_examples(self) -> List[MinesweeperExample]:
        examples: List[MinesweeperExample] = []
        for game in self.games:
            step_files = sorted(game.glob("step*.txt"), key=lambda p: int(p.stem[4:]))
            hidden_state: List[List[str]] = [
                line.split() for line in (game / "hidden_state.txt").read_text().splitlines() if line.strip()
            ]

            for i in range(len(step_files) - 1):
                state_str = step_files[i].read_text()
                board_state: List[List[str]] = [
                    line.split() for line in state_str.splitlines() if line.strip()
                ]
                examples.append(MinesweeperExample(input=state_str, board_state=board_state, hidden_state=hidden_state))
        random.shuffle(examples)
        return examples

    def to_hf_dataset(self) -> Dataset:
        """
        Convert Minesweeper examples to a Hugging Face Dataset object
        suitable for GRPO training.
        """
        examples: List[MinesweeperExample] = self.load_examples()

        hf_data: List[Dict[str, Any]] = [
            {
                "prompt": format_example(ex.input),
                "board_state": ex.board_state,
                "hidden_state": ex.hidden_state,
            }
            for ex in examples
        ]

        return Dataset.from_list(hf_data)


In [9]:
from datasets import Dataset
from pprint import pprint

train_dataset_loader = MinesweeperDatasetLoader(data_dir="train-data")
train_dataset: Dataset = train_dataset_loader.to_hf_dataset()

test_dataset_loader = MinesweeperDatasetLoader(data_dir="test-data")
test_dataset: Dataset = test_dataset_loader.to_hf_dataset()

Using data directory: /content/train-data
Data directory exists: /content/train-data
Found 90 game directories in /content/train-data
Using data directory: /content/test-data
Data directory exists: /content/test-data
Found 15 game directories in /content/test-data


In [10]:
from tqdm import tqdm

def count_tokens(example):
    # Turn messages into text using your chat template
    text = tokenizer.apply_chat_template(
        example["prompt"],  # or however your dataset stores conversations
        tokenize=False,
        add_generation_prompt=True,
    )
    # Tokenize and return number of tokens
    return len(tokenizer(text)["input_ids"])

# Collect lengths for the whole dataset
lengths = [count_tokens(example) for example in tqdm(train_dataset)]

print("Max prompt length:", max(lengths))
print("Avg prompt length:", sum(lengths) / len(lengths))
print("Some samples:", lengths[:10])

100%|██████████| 963/963 [00:01<00:00, 656.00it/s]

Max prompt length: 378
Avg prompt length: 370.32294911734164
Some samples: [377, 374, 363, 367, 360, 367, 370, 372, 373, 375]


In [11]:
from typing import List, Dict, Optional
import re
from pprint import pprint
from datasets import Dataset
from vllm import SamplingParams

max_prompt_length=420

class LLMTester:
    def __init__(self, model, tokenizer, dataset: Dataset, lora_request = None):
        self.model = model
        self.tokenizer = tokenizer
        self.dataset: Dataset = dataset
        self.lora_request = lora_request

    def test_llm(self, verbose: bool = False):
        moves_correct = 0
        for example in self.dataset:
            prompt = example["prompt"]
            board_state: List[List[str]] = example["board_state"]
            hidden_state: List[List[str]] = example["hidden_state"]

            llm_move: str = self.generate_llm_move(prompt)
            if verbose:
                print(f"Board state:")
                pprint(board_state)
                print(f"Hidden state:")
                pprint(hidden_state)
                print(f"LLM Move:\n{llm_move}")

            parsed_llm_move: Optional[Move] = self.parse_llm_move(llm_move)
            if not parsed_llm_move:
                if verbose: print(f"Error parsing llm move: {llm_move}")
                continue

            if not self.validate_llm_move(parsed_llm_move, board_state):
                if verbose: print(f"Invalid move: {parsed_llm_move}")
                continue

            if not self.verify_llm_move(parsed_llm_move, hidden_state):
                if verbose: print(f"Move incorrect: {parsed_llm_move}")
                continue

            moves_correct += 1
            if verbose: print(f"Move correct!")

        print(f"Moves correct: {moves_correct}/{len(self.dataset)}")
        print(f"Moves incorrect: {len(self.dataset) - moves_correct}/{len(self.dataset)}")

    def generate_llm_move(self, prompt: List[Dict[str, str]]) -> str:
        text = self.tokenizer.apply_chat_template(
            prompt,
            tokenize=False,
            add_generation_prompt=True,
        )
        sampling_params = SamplingParams(
            temperature=0.0,
            top_k=-1,
            top_p=1.0,
            max_tokens=max_seq_length-max_prompt_length,
        )
        output: str = model.fast_generate(
            text,
            sampling_params=sampling_params,
            lora_request=self.lora_request,
        )[0].outputs[0].text
        return output

    def parse_llm_move(self, llm_move: str) -> Optional[Move]:
        pattern = r"row:\s*(\d+),\s*col:\s*(\d+),\s*action:\s*(reveal|flag)"
        match = re.search(pattern, llm_move.strip(), re.DOTALL)
        if match:
            row, col, action = match.groups()
            return Move(row=int(row)-1, col=int(col)-1, action=action)  # 0-indexed
        return None

    def validate_llm_move(self, move: Move, board_state: List[List[str]]) -> bool:
        rows = len(board_state)
        cols = len(board_state[0])
        if not (0 <= move.row < rows and 0 <= move.col < cols):
            return False  # Out of bounds
        if board_state[move.row][move.col] != "*":  # Already revealed or flagged
            return False
        return True

    def verify_llm_move(self, move: Move, hidden_state: List[List[str]]) -> bool:
        cell_value = hidden_state[move.row][move.col]
        if move.action == "reveal":
            return cell_value != "M"  # Should not reveal a mine
        elif move.action == "flag":
            return cell_value == "M"  # Should only flag mines
        return False


In [31]:
# Example usage
llm_tester = LLMTester(model=model, tokenizer=tokenizer, dataset=test_dataset.select(range(100)))
llm_tester.test_llm(verbose=True)

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['*', '*', '*', '*', '*'],
 ['F', 'F', 'F', '*', '*'],
 ['2', '3', '2', '3', '*'],
 ['0', '0', '0', '3', '*'],
 ['0', '0', '0', '2', '*']]
Hidden state:
[['2', '3', '3', 'M', 'M'],
 ['M', 'M', 'M', '4', '3'],
 ['2', '3', '2', '3', 'M'],
 ['0', '0', '0', '3', 'M'],
 ['0', '0', '0', '2', 'M']]
LLM Move:
<think>

</think>

row: 1, col: 4, action: reveal
Move incorrect: Move(row=0, col=3, action='reveal')


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['0', '1', '*', '*', '*'],
 ['0', '2', 'F', '*', '*'],
 ['0', '3', 'F', '*', '*'],
 ['0', '2', 'F', '2', '*'],
 ['0', '1', '1', '1', '*']]
Hidden state:
[['0', '1', '2', '4', 'M'],
 ['0', '2', 'M', 'M', 'M'],
 ['0', '3', 'M', '5', '2'],
 ['0', '2', 'M', '2', '0'],
 ['0', '1', '1', '1', '0']]
LLM Move:
<think>

</think>

row: 1, col: 3, action: reveal
Move correct!


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Board state:
[['*', 'F', '2', '0', '0'],
 ['*', 'F', '3', '2', '2'],
 ['*', '1', '2', 'F', 'F'],
 ['*', '*', '*', '*', '*'],
 ['*', '*', '*', '*', '*']]
Hidden state:
[['M', 'M', '2', '0', '0'],
 ['3', 'M', '3', '2', '2'],
 ['1', '1', '2', 'M', 'M'],
 ['0', '1', '2', '5', 'M'],
 ['0', '1', 'M', '3', 'M']]
LLM Move:
<think>

</think>

row: 1, col: 5, action: reveal
Invalid move: Move(row=0, col=4, action='reveal')


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['*', '*', '*', '*', '*'],
 ['F', 'F', '3', '2', '*'],
 ['2', '2', '1', '*', '*'],
 ['0', '0', '1', '*', '*'],
 ['0', '0', '1', '*', '*']]
Hidden state:
[['M', 'M', '3', 'M', 'M'],
 ['M', 'M', '3', '2', '2'],
 ['2', '2', '1', '0', '0'],
 ['0', '0', '1', '1', '1'],
 ['0', '0', '1', 'M', '1']]
LLM Move:
<think>

</think>

row: 1, col: 4, action: reveal
Move incorrect: Move(row=0, col=3, action='reveal')


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['F', '2', '0', '0', '0'],
 ['F', '3', '1', '1', '0'],
 ['*', '*', '*', '3', '2'],
 ['*', '*', '*', '*', '*'],
 ['*', '*', '*', '*', '*']]
Hidden state:
[['M', '2', '0', '0', '0'],
 ['M', '3', '1', '1', '0'],
 ['1', '2', 'M', '3', '2'],
 ['0', '2', '3', 'M', 'M'],
 ['0', '1', 'M', '3', '2']]
LLM Move:
<think>

</think>

row: 3, col: 1, action: reveal
Move correct!
Moves correct: 51/100
Moves incorrect: 49/100


In [32]:
from huggingface_hub import snapshot_download

# Download the repo contents into a local folder
local_dir = snapshot_download("adi-256/minesweepergpt")
print(local_dir)

# Now load from local path
lora_request = model.load_lora(local_dir)

llm_tester = LLMTester(model, tokenizer, test_dataset.select(range(100)), lora_request=lora_request)
llm_tester.test_llm(verbose=True)

/root/.cache/huggingface/hub/models--adi-256--minesweepergpt/snapshots/b5137eb1b92fc2d59eda480ddb5f69c093abba21


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['*', '*', '*', '*', '*'],
 ['F', 'F', 'F', '*', '*'],
 ['2', '3', '2', '3', '*'],
 ['0', '0', '0', '3', '*'],
 ['0', '0', '0', '2', '*']]
Hidden state:
[['2', '3', '3', 'M', 'M'],
 ['M', 'M', 'M', '4', '3'],
 ['2', '3', '2', '3', 'M'],
 ['0', '0', '0', '3', 'M'],
 ['0', '0', '0', '2', 'M']]
LLM Move:
<think>

</think>

row: 1, col: 4, action: reveal
Move incorrect: Move(row=0, col=3, action='reveal')


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['0', '1', '*', '*', '*'],
 ['0', '2', 'F', '*', '*'],
 ['0', '3', 'F', '*', '*'],
 ['0', '2', 'F', '2', '*'],
 ['0', '1', '1', '1', '*']]
Hidden state:
[['0', '1', '2', '4', 'M'],
 ['0', '2', 'M', 'M', 'M'],
 ['0', '3', 'M', '5', '2'],
 ['0', '2', 'M', '2', '0'],
 ['0', '1', '1', '1', '0']]
LLM Move:
<think>

</think>

row: 1, col: 3, action: reveal
Move correct!


...

Board state:
[['*', 'F', '2', '0', '0'],
 ['*', 'F', '3', '2', '2'],
 ['*', '1', '2', 'F', 'F'],
 ['*', '*', '*', '*', '*'],
 ['*', '*', '*', '*', '*']]
Hidden state:
[['M', 'M', '2', '0', '0'],
 ['3', 'M', '3', '2', '2'],
 ['1', '1', '2', 'M', 'M'],
 ['0', '1', '2', '5', 'M'],
 ['0', '1', 'M', '3', 'M']]
LLM Move:
<think>

</think>

row: 1, col: 5, action: reveal
Invalid move: Move(row=0, col=4, action='reveal')


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['*', '*', '*', '*', '*'],
 ['F', 'F', '3', '2', '*'],
 ['2', '2', '1', '*', '*'],
 ['0', '0', '1', '*', '*'],
 ['0', '0', '1', '*', '*']]
Hidden state:
[['M', 'M', '3', 'M', 'M'],
 ['M', 'M', '3', '2', '2'],
 ['2', '2', '1', '0', '0'],
 ['0', '0', '1', '1', '1'],
 ['0', '0', '1', 'M', '1']]
LLM Move:
<think>

</think>

row: 1, col: 1, action: reveal
Move incorrect: Move(row=0, col=0, action='reveal')


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['F', '2', '0', '0', '0'],
 ['F', '3', '1', '1', '0'],
 ['*', '*', '*', '3', '2'],
 ['*', '*', '*', '*', '*'],
 ['*', '*', '*', '*', '*']]
Hidden state:
[['M', '2', '0', '0', '0'],
 ['M', '3', '1', '1', '0'],
 ['1', '2', 'M', '3', '2'],
 ['0', '2', '3', 'M', 'M'],
 ['0', '1', 'M', '3', '2']]
LLM Move:
<think>

</think>

row: 3, col: 1, action: reveal
Move correct!
Moves correct: 58/100
Moves incorrect: 42/100
